<a href="https://colab.research.google.com/github/juanfebotero/juanfebotero.github.io/blob/main/notebooks/future-internet/02_virtual_network_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Virtual Network Embedding: un algoritmo voraz paso a paso

**Internet del Futuro: Redes Programables** · Prof. Juan Felipe Botero · Universidad de Antioquia

**Objetivos**

- Entender el problema de *Virtual Network Embedding* (VNE) y por qué es NP-difícil.
- Implementar un mapeo en dos etapas: nodos y luego enlaces.
- Medir la tasa de aceptación de solicitudes.
- Diseñar una mejor función de ranking de nodos.

> **Antes de empezar:** use *Archivo → Guardar una copia en Drive*. Si el cuaderno le llegó por Google Classroom, ya es su copia personal y el profesor verá su progreso.

## 1. El problema
Un proveedor de infraestructura tiene una **red sustrato** $G_S$ con CPU en nodos y ancho de banda en enlaces. Llegan **solicitudes de red virtual** $G_V$. Hay que asignar cada nodo virtual a un nodo físico distinto con CPU suficiente, y cada enlace virtual a un camino físico con ancho de banda suficiente.

Referencia: A. Fischer, **J. F. Botero**, M. T. Beck, H. de Meer, X. Hesselbach. *Virtual Network Embedding: A Survey*. IEEE COMST, 2013.

In [ ]:
import random, networkx as nx

def red_sustrato(n=20, p=0.2, semilla=1):
    rng = random.Random(semilla)
    G = nx.connected_watts_strogatz_graph(n, 4, p, seed=semilla)
    for v in G.nodes: G.nodes[v]["cpu"] = rng.randint(50, 100)
    for u, v in G.edges: G[u][v]["bw"] = rng.randint(50, 100)
    return G

def solicitud(rng, nmin=2, nmax=5):
    n = rng.randint(nmin, nmax)
    R = nx.gnp_random_graph(n, 0.5, seed=rng.randint(0, 10**6))
    while not nx.is_connected(R):
        R = nx.gnp_random_graph(n, 0.5, seed=rng.randint(0, 10**6))
    for v in R.nodes: R.nodes[v]["cpu"] = rng.randint(5, 30)
    for u, w in R.edges: R[u][w]["bw"] = rng.randint(5, 30)
    return R

def mapear(G, R, rango=lambda G, v: G.nodes[v]["cpu"]):
    """Mapeo voraz en dos etapas. Devuelve (mapa_nodos, mapa_enlaces) o None si se rechaza."""
    libres = sorted(G.nodes, key=lambda v: rango(G, v), reverse=True)
    mn = {}
    for r in sorted(R.nodes, key=lambda r: R.nodes[r]["cpu"], reverse=True):
        cand = [v for v in libres if v not in mn.values() and G.nodes[v]["cpu"] >= R.nodes[r]["cpu"]]
        if not cand: return None
        mn[r] = cand[0]
    H = G.copy(); me = {}
    for a, b in R.edges:
        bw = R[a][b]["bw"]
        sub = nx.subgraph_view(H, filter_edge=lambda u, v: H[u][v]["bw"] >= bw)
        try: camino = nx.shortest_path(sub, mn[a], mn[b])
        except nx.NetworkXNoPath: return None
        for u, v in zip(camino, camino[1:]): H[u][v]["bw"] -= bw
        me[(a, b)] = camino
    for r, v in mn.items(): G.nodes[v]["cpu"] -= R.nodes[r]["cpu"]
    for (a, b), camino in me.items():
        for u, v in zip(camino, camino[1:]): G[u][v]["bw"] -= R[a][b]["bw"]
    return mn, me

def experimento(k=60, rango=lambda G, v: G.nodes[v]["cpu"], semilla=7):
    G = red_sustrato(); rng = random.Random(semilla); aceptadas = 0
    for _ in range(k):
        if mapear(G, solicitud(rng), rango) is not None: aceptadas += 1
    return aceptadas

In [ ]:
import matplotlib.pyplot as plt
G = red_sustrato(); rng = random.Random(0); R = solicitud(rng)
res = mapear(G.copy(), R)
print("Mapa de nodos:", res[0]); print("Mapa de enlaces:", res[1])
pos = nx.spring_layout(G, seed=2)
usados = set(res[0].values())
nx.draw(G, pos, node_color=["#0b6b45" if v in usados else "#dcefe4" for v in G.nodes], with_labels=True, node_size=350, font_size=8)
plt.title("Nodos físicos que alojan la red virtual"); plt.show()

## 2. Tasa de aceptación
Llegan 60 solicitudes en secuencia (sin que expiren). ¿Cuántas acepta el algoritmo voraz?

In [ ]:
print(experimento(), 'de 60 solicitudes aceptadas')

## 3. Ejercicios
**Ejercicio 1.** Guarde en `resp_1` el número de solicitudes aceptadas por `experimento()` con el ranking por CPU.

In [ ]:
resp_1 = None

**Ejercicio 2.** El ranking por CPU ignora el ancho de banda. Pruebe el ranking de Zhu y Ammar: $H(v) = \text{CPU}(v)\cdot\sum_{e \ni v} \text{BW}(e)$. Escriba la función `rango_h(G, v)`, ejecute `experimento(rango=rango_h)` y compare. ¿Mejora la aceptación? ¿Por qué?

In [ ]:
def rango_h(G, v):
    # complete aquí
    return G.nodes[v]['cpu']

print(experimento(rango=rango_h))

**Ejercicio 3 (reto).** Haga que las solicitudes expiren después de 10 llegadas (libere sus recursos) y grafique la aceptación acumulada en el tiempo para ambos rankings.

In [ ]:
# --- Autoevaluación: no modifique esta celda ---
import math
def verificar(nombre, valor, esperado, tol=1e-6):
    """Compara su respuesta con la esperada y muestra el resultado."""
    if valor is None:
        print(f"[ ] {nombre}: pendiente, complete la celda anterior.")
        return False
    try:
        ok = abs(float(valor) - float(esperado)) <= tol * max(1, abs(float(esperado)))
    except (TypeError, ValueError):
        ok = valor == esperado
    print(f"[{'✔' if ok else '✘'}] {nombre}: {'correcto' if ok else 'revise su respuesta (obtuvo ' + repr(valor) + ')'}")
    return ok

verificar('Ejercicio 1', resp_1, 15)